# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
import os

if not os.path.exists('requirements.txt'):
    !wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt -q


requirements.txt already exists.
Note: you may need to restart the kernel to use updated packages.


## 2. Set API Keys

In [2]:
import os
from dotenv import load_dotenv

# Fallback: try Google Colab secrets
try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
except ImportError:
    # Load from .env file (local development)
    load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
GEMINI_API_KEY = (os.getenv("GEMINI_API_KEY") or "").strip()


if not GEMINI_API_KEY or not HF_TOKEN:
    print("Warning: HF_TOKEN or GEMINI_API_KEY not set. Please set it in .env file or Colab secrets.")

In [3]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

# Force anonymous download for public embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_138695/1290443477.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector DB created successfully (8 chunks)


In [5]:
# Create retriever — returns top 2 most similar documents for any query
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print("Retriever ready.")

Retriever ready.


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = retriever.invoke(query)
    docs = "\n".join([doc.page_content for doc in results])
    return StrOutputParser().invoke(docs)

# Switch providers via env vars (set before running this cell):
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash  (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   ollama       → MODEL_NAME=ollama:llama3
#   huggingface  → MODEL_NAME= huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0
#MODEL_NAME = "google_genai:gemini-2.5-flash"
#model_api_key = GEMINI_API_KEY
MODEL_NAME = "huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model_api_key = HF_TOKEN

chat_llm = init_chat_model(
    MODEL_NAME,
    api_key=model_api_key
)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "Your goal is to answer the user's question. "
        "First, use the 'search_answers' tool to retrieve relevant information from the knowledge base. "
        "Then, use the retrieved information to formulate a comprehensive and concise answer to the user's question. "
        "Always provide a final answer to the user after using tools. Never return an empty response."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

Agent ready (model: google_genai:gemini-2.5-flash-lite).


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [10]:
def ask(question: str) -> str:
    response = agent.invoke({"messages": [{"role": "user", "content": question}]})
    # Get the last message (final answer from agent)
    return response["messages"][-1].content

print(ask("What is DQE?"))



<|system|>
Your goal is to answer the user's question. First, use the 'search_answers' tool to retrieve relevant information from the knowledge base. Then, use the retrieved information to formulate a comprehensive and concise answer to the user's question. Always provide a final answer to the user after using tools. Never return an empty response.</s>
<|user|>
What is DQE?</s>
<|assistant|>
DQE or Decision-Enhanced Quality Engineering is a philosophy and approach to product development that aims to enhance the quality of the products by combining the principles of quality engineering with the focus on decision-making.

DQE emphasizes the importance of decision-making in the product development process and encourages the use of metrics and principles of quality engineering to help the development team make informed decisions about product design, design process, and system architecture.

DQE supports the development team in making informed decisions by providing tools and techniques to

In [8]:
print(ask("How many students does DQE have?"))


Agent's Answer: <|system|>
Your goal is to answer the user's question. First, use the 'search_answers' tool to retrieve relevant information from the knowledge base. Then, use the retrieved information to formulate a comprehensive and concise answer to the user's question. If the search results do not contain enough information to answer the question, state that you cannot answer the question based on the provided knowledge base.</s>
<|user|>
How many students does DQE have?</s>
<|assistant|>
I do not have access to the latest information about dqe's current student population. However, based on the company's website, dqe is a private, non-profit institution that provides education, training, and vocational services to individuals in the greater los angeles area, including california and nevada. You may visit their website or contact their human resources department for more information on their current enrollment and student population.


In [9]:
print(ask("Whats the objective of DQE department"))


Agent's Answer: <|system|>
Your goal is to answer the user's question. First, use the 'search_answers' tool to retrieve relevant information from the knowledge base. Then, use the retrieved information to formulate a comprehensive and concise answer to the user's question. If the search results do not contain enough information to answer the question, state that you cannot answer the question based on the provided knowledge base.</s>
<|user|>
Whats the objective of DQE department</s>
<|assistant|>
The objective of DQE department is to provide customers with an efficient and reliable supply chain management solution. The department works to improve the overall effectiveness of the supply chain by ensuring that all processes are optimized, streamlined, and integrated seamlessly.

The department's primary focus is on reducing costs, improving efficiency, and increasing accuracy in order fulfillment. This is accomplished through the implementation of best practices, continuous improvement